# 11 - Read Azure setup

Read subscription state, service provider registration, region lists, VM quota, and installed CLI extensions. The candidate region is `southeastasia` in `configs/azure.json`; it is not a confirmed deployment choice. This notebook does not create resources, register providers, install extensions, or print account IDs and tokens.


In [1]:
from pathlib import Path
import json
import platform
import subprocess
import sys

assert platform.system() == "Linux", "Use the WSL kernel."
assert Path(sys.prefix).name == "mlop-pj", "Select the project environment."
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())
config = json.loads((ROOT / "configs/azure.json").read_text())
print("Candidate region:", config["candidate_region"])
print("Budget before credits:", config["budget_thb_before_credits"], "THB")


Candidate region: southeastasia
Budget before credits: 150 THB


## Read the checks

Run this cell, then save the notebook. The checks may take a minute. Review missing providers, region support, and failed checks. VM quota is only a starting point; Azure ML and Container Apps quotas still need their own review. Permissions, current prices, and cleanup timing also remain open, so `deploy_ready` stays false. This is a setup report, not a failed model test.

The earlier error below came from quota counts returned as strings. The script now converts valid integer strings before subtraction. Rerun this code cell to get the corrected report; the old output is kept for review.


In [2]:
result = subprocess.run([sys.executable, "cloudlayer/preflight.py"],
                        cwd=ROOT, capture_output=True, text=True)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Preflight failed. Read the Python traceback above.")
report = json.loads(result.stdout)
print(json.dumps({key: report.get(key) for key in ["candidate_region", "subscription_enabled",
                  "region_listed_for_subscription", "providers", "vm_quota", "extensions",
                  "checks_failed", "deploy_ready", "next"]}, indent=2))


AssertionError: Preflight script failed; keep the output for review.

## Output

`reports/azure-preflight.json`. Send the saved results from 10 and 11. Cloud creation comes after cost, permissions, and service quota review.
